# E-Commerce Intelligence Platform — Exploratory Data Analysis
**Dataset:** Olist Brazilian E-Commerce (Kaggle)  
**Author:** Ramu Battu — MS Data Analytics, CSU Fresno  
**Purpose:** EDA on 100K+ orders before ML modeling and AWS pipeline

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('husl')

print('Libraries loaded ✅')

## 2. Load Datasets

In [ ]:
DATA_PATH = '../data/raw/'

orders    = pd.read_csv(DATA_PATH + 'olist_orders_dataset.csv')
items     = pd.read_csv(DATA_PATH + 'olist_order_items_dataset.csv')
customers = pd.read_csv(DATA_PATH + 'olist_customers_dataset.csv')
payments  = pd.read_csv(DATA_PATH + 'olist_order_payments_dataset.csv')
reviews   = pd.read_csv(DATA_PATH + 'olist_order_reviews_dataset.csv')
products  = pd.read_csv(DATA_PATH + 'olist_products_dataset.csv')
sellers   = pd.read_csv(DATA_PATH + 'olist_sellers_dataset.csv')

print('Dataset shapes:')
for name, df in [('orders',orders),('items',items),('customers',customers),
                 ('payments',payments),('reviews',reviews),('products',products),('sellers',sellers)]:
    print(f'  {name:12s}: {df.shape[0]:>7,} rows × {df.shape[1]} cols')

## 3. Orders — Basic Overview

In [ ]:
print('Orders shape:', orders.shape)
print('\nOrder status distribution:')
print(orders['order_status'].value_counts())
print('\nMissing values:')
print(orders.isnull().sum()[orders.isnull().sum() > 0])

## 4. Date Parsing & Feature Engineering

In [ ]:
date_cols = ['order_purchase_timestamp','order_approved_at',
             'order_delivered_carrier_date','order_delivered_customer_date',
             'order_estimated_delivery_date']
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

orders['year']          = orders['order_purchase_timestamp'].dt.year
orders['month']         = orders['order_purchase_timestamp'].dt.month
orders['day_of_week']   = orders['order_purchase_timestamp'].dt.day_name()
orders['hour']          = orders['order_purchase_timestamp'].dt.hour
orders['delivery_days'] = (orders['order_delivered_customer_date'] -
                           orders['order_purchase_timestamp']).dt.days
orders['is_late']       = (orders['order_delivered_customer_date'] >
                           orders['order_estimated_delivery_date']).astype(int)

orders_delivered = orders[orders['order_status'] == 'delivered'].copy()
print(f'Delivered orders: {len(orders_delivered):,} ({len(orders_delivered)/len(orders)*100:.1f}%)')
print(f'Late deliveries:  {orders_delivered["is_late"].sum():,} ({orders_delivered["is_late"].mean()*100:.1f}%)')
print(f'Avg delivery days: {orders_delivered["delivery_days"].mean():.1f}')

## 5. Monthly Order Volume & Revenue Trend

In [ ]:
# Join payments to get revenue
pay_agg = payments.groupby('order_id')['payment_value'].sum().reset_index()
pay_agg.columns = ['order_id','total_payment']
orders_pay = orders_delivered.merge(pay_agg, on='order_id', how='left')

monthly = orders_pay.groupby(['year','month']).agg(
    order_count=('order_id','count'),
    revenue=('total_payment','sum')
).reset_index()
monthly['period'] = pd.to_datetime(monthly[['year','month']].assign(day=1))
monthly = monthly[monthly['period'] < '2018-09-01']  # Remove incomplete months

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

ax1.bar(monthly['period'], monthly['order_count'], color='steelblue', alpha=0.8)
ax1.set_title('Monthly Order Volume', fontsize=14, fontweight='bold')
ax1.set_xlabel('Month'); ax1.set_ylabel('Orders')
ax1.tick_params(axis='x', rotation=45)

ax2.bar(monthly['period'], monthly['revenue']/1e6, color='coral', alpha=0.8)
ax2.set_title('Monthly Revenue (BRL Millions)', fontsize=14, fontweight='bold')
ax2.set_xlabel('Month'); ax2.set_ylabel('Revenue (M BRL)')
ax2.tick_params(axis='x', rotation=45)

plt.suptitle('Olist E-Commerce — Monthly Trends (2016–2018)', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../dashboard/eda_monthly_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Peak month: {monthly.loc[monthly["order_count"].idxmax(), "period"].strftime("%B %Y")} ({monthly["order_count"].max():,} orders)')

## 6. Order Status Distribution

In [ ]:
status_counts = orders['order_status'].value_counts()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#2ecc71','#e74c3c','#f39c12','#3498db','#9b59b6','#1abc9c','#e67e22','#95a5a6']
ax1.pie(status_counts.values, labels=status_counts.index, autopct='%1.1f%%',
        colors=colors[:len(status_counts)], startangle=90,
        wedgeprops=dict(edgecolor='white', linewidth=2))
ax1.set_title('Order Status Distribution', fontsize=13, fontweight='bold')

ax2.barh(status_counts.index, status_counts.values, color=colors[:len(status_counts)])
ax2.set_title('Order Status Count', fontsize=13, fontweight='bold')
ax2.set_xlabel('Number of Orders')
for i, v in enumerate(status_counts.values):
    ax2.text(v + 100, i, f'{v:,}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../dashboard/eda_order_status.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Delivery Analysis

In [ ]:
delivery_clean = orders_delivered.dropna(subset=['delivery_days'])
delivery_clean = delivery_clean[delivery_clean['delivery_days'].between(0, 60)]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Delivery days distribution
axes[0].hist(delivery_clean['delivery_days'], bins=40, color='steelblue', alpha=0.8, edgecolor='white')
axes[0].axvline(delivery_clean['delivery_days'].mean(), color='red', linestyle='--', label=f"Mean: {delivery_clean['delivery_days'].mean():.1f} days")
axes[0].set_title('Delivery Days Distribution', fontweight='bold')
axes[0].set_xlabel('Days'); axes[0].set_ylabel('Orders')
axes[0].legend()

# Late vs On-time
late_counts = delivery_clean['is_late'].value_counts()
axes[1].bar(['On-Time','Late'], [late_counts.get(0,0), late_counts.get(1,0)],
            color=['#2ecc71','#e74c3c'], alpha=0.85, edgecolor='white')
axes[1].set_title('On-Time vs Late Deliveries', fontweight='bold')
axes[1].set_ylabel('Orders')
for i, v in enumerate([late_counts.get(0,0), late_counts.get(1,0)]):
    axes[1].text(i, v+100, f'{v:,}\n({v/len(delivery_clean)*100:.1f}%)', ha='center', fontweight='bold')

# Avg delivery by day of week
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow_avg = delivery_clean.groupby('day_of_week')['delivery_days'].mean().reindex(dow_order)
axes[2].bar(range(7), dow_avg.values, color='coral', alpha=0.85, edgecolor='white')
axes[2].set_xticks(range(7))
axes[2].set_xticklabels([d[:3] for d in dow_order])
axes[2].set_title('Avg Delivery Days by Order Day', fontweight='bold')
axes[2].set_ylabel('Avg Days')

plt.suptitle('Delivery Performance Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../dashboard/eda_delivery_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Late delivery rate: {delivery_clean["is_late"].mean()*100:.1f}%')
print(f'Avg delivery time: {delivery_clean["delivery_days"].mean():.1f} days')

## 8. Payment Analysis

In [ ]:
pay_type = payments.groupby('payment_type').agg(
    count=('order_id','count'),
    total_value=('payment_value','sum'),
    avg_value=('payment_value','mean')
).reset_index().sort_values('count', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#3498db','#e74c3c','#2ecc71','#f39c12','#9b59b6']
axes[0].bar(pay_type['payment_type'], pay_type['count'], color=colors[:len(pay_type)], alpha=0.85)
axes[0].set_title('Payment Method — Order Count', fontweight='bold')
axes[0].set_xlabel('Payment Type'); axes[0].set_ylabel('Orders')
for i, (_, row) in enumerate(pay_type.iterrows()):
    axes[0].text(i, row['count']+200, f'{row["count"]:,}', ha='center', fontsize=9, fontweight='bold')

axes[1].bar(pay_type['payment_type'], pay_type['total_value']/1e6, color=colors[:len(pay_type)], alpha=0.85)
axes[1].set_title('Payment Method — Total Revenue (M BRL)', fontweight='bold')
axes[1].set_xlabel('Payment Type'); axes[1].set_ylabel('Revenue (M BRL)')

plt.tight_layout()
plt.savefig('../dashboard/eda_payment_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Payment summary:')
print(pay_type[['payment_type','count','total_value','avg_value']].to_string(index=False))

## 9. Review Score Distribution

In [ ]:
review_dist = reviews['review_score'].value_counts().sort_index()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

score_colors = ['#e74c3c','#e67e22','#f1c40f','#2ecc71','#27ae60']
ax1.bar(review_dist.index, review_dist.values, color=score_colors, alpha=0.9, edgecolor='white')
ax1.set_title('Review Score Distribution', fontweight='bold')
ax1.set_xlabel('Review Score (1-5)'); ax1.set_ylabel('Number of Reviews')
for i, v in zip(review_dist.index, review_dist.values):
    ax1.text(i, v+100, f'{v:,}\n({v/len(reviews)*100:.1f}%)', ha='center', fontsize=9, fontweight='bold')

# Review score by delivery late/on-time
merged_rev = orders_delivered[['order_id','is_late']].merge(
    reviews[['order_id','review_score']], on='order_id', how='inner')
rev_by_late = merged_rev.groupby(['is_late','review_score']).size().unstack(fill_value=0)
rev_by_late.index = ['On-Time','Late']
rev_by_late.plot(kind='bar', ax=ax2, color=score_colors, alpha=0.85, edgecolor='white')
ax2.set_title('Review Score: On-Time vs Late Delivery', fontweight='bold')
ax2.set_xlabel('Delivery Status'); ax2.set_ylabel('Count')
ax2.legend(title='Score', bbox_to_anchor=(1.01,1))
ax2.tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('../dashboard/eda_review_scores.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Average review score: {reviews["review_score"].mean():.2f}/5.0')
print(f'5-star reviews: {(reviews["review_score"]==5).sum():,} ({(reviews["review_score"]==5).mean()*100:.1f}%)')

## 10. Top States by Orders & Revenue

In [ ]:
cust_orders = orders_pay.merge(customers[['customer_id','customer_state']], on='customer_id', how='left')
state_stats = cust_orders.groupby('customer_state').agg(
    orders=('order_id','count'),
    revenue=('total_payment','sum'),
    avg_delivery=('delivery_days','mean')
).reset_index().sort_values('orders', ascending=False)

top10 = state_stats.head(10)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].barh(top10['customer_state'][::-1], top10['orders'][::-1], color='steelblue', alpha=0.85)
axes[0].set_title('Top 10 States — Order Volume', fontweight='bold')
axes[0].set_xlabel('Orders')
for i, (_, row) in enumerate(top10[::-1].iterrows()):
    axes[0].text(row['orders']+100, i, f'{row["orders"]:,}', va='center', fontsize=9)

axes[1].barh(top10['customer_state'][::-1], top10['revenue'][::-1]/1e6, color='coral', alpha=0.85)
axes[1].set_title('Top 10 States — Revenue (M BRL)', fontweight='bold')
axes[1].set_xlabel('Revenue (M BRL)')

plt.suptitle('Geographic Distribution — Top 10 States', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../dashboard/eda_top_states.png', dpi=150, bbox_inches='tight')
plt.show()
print('Top 5 states:')
print(top10[['customer_state','orders','revenue']].head().to_string(index=False))

## 11. Hourly Order Pattern

In [ ]:
hourly = orders.groupby('hour').size().reset_index(name='orders')

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(hourly['hour'], hourly['orders'],
              color=['#e74c3c' if h in hourly.nlargest(3,'orders')['hour'].values else 'steelblue'
                     for h in hourly['hour']],
              alpha=0.85, edgecolor='white')
ax.set_title('Hourly Order Pattern — Peak Shopping Hours', fontsize=14, fontweight='bold')
ax.set_xlabel('Hour of Day (0–23)')
ax.set_ylabel('Number of Orders')
ax.set_xticks(range(24))
ax.axvspan(10, 16, alpha=0.1, color='red', label='Peak hours')
ax.legend()
peak_hour = hourly.loc[hourly['orders'].idxmax(), 'hour']
ax.annotate(f'Peak: {peak_hour}:00', xy=(peak_hour, hourly['orders'].max()),
            xytext=(peak_hour+2, hourly['orders'].max()*0.95),
            arrowprops=dict(arrowstyle='->', color='red'), color='red', fontweight='bold')
plt.tight_layout()
plt.savefig('../dashboard/eda_hourly_pattern.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Peak hour: {peak_hour}:00 with {hourly["orders"].max():,} orders')

## 12. RFM Feature Preview (for ML pipeline)

In [ ]:
snapshot = orders['order_purchase_timestamp'].max()
cust_orders2 = orders_pay.dropna(subset=['total_payment'])

rfm = cust_orders2.groupby('customer_id').agg(
    recency=('order_purchase_timestamp', lambda x: (snapshot - x.max()).days),
    frequency=('order_id', 'count'),
    monetary=('total_payment', 'sum')
).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(rfm['recency'].clip(upper=500), bins=50, color='#3498db', alpha=0.8, edgecolor='white')
axes[0].set_title('Recency Distribution (days)', fontweight='bold')
axes[0].set_xlabel('Days since last order')

axes[1].hist(rfm['frequency'].clip(upper=10), bins=20, color='#2ecc71', alpha=0.8, edgecolor='white')
axes[1].set_title('Frequency Distribution', fontweight='bold')
axes[1].set_xlabel('Number of orders')

axes[2].hist(rfm['monetary'].clip(upper=1000), bins=50, color='#e67e22', alpha=0.8, edgecolor='white')
axes[2].set_title('Monetary Distribution (BRL)', fontweight='bold')
axes[2].set_xlabel('Total spend (BRL)')

plt.suptitle('RFM Feature Distributions — Input for Customer Segmentation ML', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../dashboard/eda_rfm_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('RFM Summary:')
print(rfm[['recency','frequency','monetary']].describe().round(2))

## 13. Key EDA Findings — Summary

In [ ]:
print('=' * 60)
print('  EDA SUMMARY — OLIST BRAZILIAN E-COMMERCE')
print('=' * 60)
print(f'  Total orders:          {len(orders):>10,}')
print(f'  Delivered orders:      {len(orders_delivered):>10,}')
print(f'  Unique customers:      {customers["customer_unique_id"].nunique():>10,}')
print(f'  Unique sellers:        {sellers.shape[0]:>10,}')
print(f'  Avg review score:      {reviews["review_score"].mean():>10.2f} / 5.0')
print(f'  Late delivery rate:    {delivery_clean["is_late"].mean()*100:>9.1f}%')
print(f'  Avg delivery days:     {delivery_clean["delivery_days"].mean():>10.1f}')
print(f'  Top state by orders:   {state_stats.iloc[0]["customer_state"]:>10}')
print(f'  Peak order hour:       {peak_hour:>9}:00')
print(f'  Most used payment:     {pay_type.iloc[0]["payment_type"]:>10}')
print('=' * 60)
print('\nKey Insights for ML Pipeline:')
print('  • High recency skew → most customers ordered once (RFM segmentation needed)')
print('  • Late deliveries correlate with lower review scores')
print('  • SP state dominates — important feature for demand forecasting')
print('  • Peak hours 10-16 → timing feature for ML models')
print('  • Credit card dominates payments → payment type is a useful feature')